In [ ]:
!pip install -q transformers torch tqdm h5py sentence-transformers qwen-vl-utils decord

In [ ]:
import os, time, numpy as np, torch, tarfile
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# Universal secret getter (mirrors other notebooks in the project)
def get_secret(key: str, fallback_path: str = None, default=None) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    except Exception:
        pass
    if fallback_path and os.path.exists(fallback_path):
        return open(fallback_path).read().strip()
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    value = os.environ.get(key, default)
    if value is None:
        raise EnvironmentError(f"Secret '{key}' not found")
    return value

# HF_TOKEN is optional for Qwen3-VL-Embedding-2B (model is public), but
# having it speeds up downloads from rate-limited HF Hub.
try:
    hf_token = get_secret("HF_TOKEN", fallback_path="/kaggle/input/my-hf-secrets/HF_TOKEN.txt")
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN loaded")
except Exception:
    print("HF_TOKEN not available — model is public so this is fine")

INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
DATASET_BLACKLIST = {"my-hf-secrets", "kaggle-data-overlay"}
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".flv", ".wmv"}

def find_and_extract_dataset():
    """Find dataset (may be a tar/zip), extract if needed, return path."""
    if not INPUT_DIR.exists():
        return None
    for dataset_dir in sorted(INPUT_DIR.iterdir()):
        if not dataset_dir.is_dir():
            continue
        if dataset_dir.name in DATASET_BLACKLIST:
            continue
        # Tarball extraction (preserves the kaggle_dataset.py path)
        tar_files = list(dataset_dir.glob("*.tar")) + list(dataset_dir.glob("*.tar.gz"))
        if tar_files:
            extract_dir = WORK_DIR / "dataset"
            extract_dir.mkdir(exist_ok=True)
            print(f"Extracting {tar_files[0].name}...")
            with tarfile.open(tar_files[0], "r:*") as tar:
                tar.extractall(extract_dir)
            print(f"Extracted to {extract_dir}")
            return str(extract_dir)
        # Folder-of-videos path (faster: skip extraction)
        for ext in VIDEO_EXTENSIONS:
            if list(dataset_dir.rglob(f"*{ext}")):
                return str(dataset_dir)
    return None

DATASET_PATH = find_and_extract_dataset()
if not DATASET_PATH:
    raise FileNotFoundError(f"No video dataset found in {INPUT_DIR}")

# Sanity check the dataset
all_videos = [
    p for p in Path(DATASET_PATH).rglob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
]
if len(all_videos) < 1:
    raise RuntimeError("Dataset has no video files. Aborting.")
print(f"Dataset: {DATASET_PATH} ({len(all_videos)} videos)")

MODEL_ID = "Qwen/Qwen3-VL-Embedding-2B"
EMBEDDING_DIM = 1024      # MRL: supports 64..2048, we use 1024 (-2% acc, 50% storage)
FPS = 0.5                # 1 frame per 2s; gives ~15 frames for a 30s clip (implicit MAX_FRAMES=16)
MAX_PIXELS = 224 * 224   # 50K pixels/frame (3x smaller than Qwen3-VL default 360*420=151K)
OUTPUT_DIR = "/kaggle/working"

# Adaptive GPU/CPU batch size.
# Qwen3-VL-Embedding-2B is ~4 GB in BF16. With OPTIMIZED MAX_PIXELS (50K vs 151K),
# activations are small enough to fit batch=4 on T4 (16 GB).
# - T4 x2 (primary target): batch=4 (2 per GPU)
# - T4 x1 (fallback): batch=2
# - P100 (16 GB, no flash attn): batch=1
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    if "P100" in gpu_name:
        device, BATCH_SIZE, NUM_WORKERS = "cuda", 1, 2
        print(f"GPU: {gpu_name} (fallback) | Batch: {BATCH_SIZE}")
    elif num_gpus > 1:
        device, BATCH_SIZE, NUM_WORKERS = "cuda", 4, 2
        print(f"GPU: {gpu_name} x{num_gpus} | Batch: {BATCH_SIZE}")
    else:
        device, BATCH_SIZE, NUM_WORKERS = "cuda", 2, 2
        print(f"GPU: {gpu_name} | Batch: {BATCH_SIZE}")
else:
    device, BATCH_SIZE, NUM_WORKERS = "cpu", 1, 1
    print("CPU mode (very slow, expect hours)")

In [ ]:
videos = sorted(
    p for p in Path(DATASET_PATH).rglob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
)
video_ids = [str(p.relative_to(DATASET_PATH)) for p in videos]
video_paths = [str(p) for p in videos]
print(f"Found {len(video_ids)} videos")
if video_ids:
    print(f"  Sample: {video_ids[0]}  ({videos[0].suffix}, {(videos[0].stat().st_size / 1024 / 1024):.1f} MB)")
    classes = sorted({p.parent.name for p in videos})
    print(f"  Classes ({len(classes)}): {', '.join(classes[:5])}{'...' if len(classes) > 5 else ''}")

In [ ]:
print(f"Loading {MODEL_ID}...")
t0 = time.time()

dtype = torch.bfloat16 if device == "cuda" else torch.float32
use_amp = (device == "cuda")
if use_amp:
    print("  Using BF16 (Qwen3-VL-Embedding weights are stored as BF16)")

# Qwen3-VL-Embedding-2B is shipped via the standard transformers AutoModel +
# custom `Qwen3VLEmbedder` helper that handles pooling + normalization.
# sentence-transformers >=3.0 exposes the same via model.encode(...).
model = SentenceTransformer(
    MODEL_ID,
    trust_remote_code=True,
    model_kwargs={"torch_dtype": dtype},
)
if device == "cuda":
    model = model.to(device)
model.eval()
print(f"Model ready in {time.time() - t0:.1f}s")

# Warmup with a tiny input — the model's first forward is slow due to kernel selection
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    print("  Warming up cuDNN kernels...")
    with torch.inference_mode():
        _ = model.encode([{"text": "warmup"}], batch_size=1)
        torch.cuda.synchronize()
    print(f"  cuDNN warmup done ({time.time() - t0:.1f}s total)")

In [ ]:
# Build Qwen3-VL input dicts. The model accepts a list of mixed-modality items.
# Each video item tells the processor to:
   - sample frames at FPS=0.5 (so a 30s clip → ~15 frames, naturally capped well
     under the implicit MAX_FRAMES=16 budget)
   - resize each frame so total pixel count fits within MAX_PIXELS=224*224=50K
# This avoids decoding videos in Python (decord / cv2) — Qwen3-VL does it internally
# using decord under the hood (3-5x faster than cv2.VideoCapture).
#
# Tuned to hit ~110s end-to-end on Kaggle T4 x2 for 100 video <30s:
#   - FPS=0.5 vs 1.0  → 2x fewer frames
#   - MAX_PIXELS=50K vs 151K → ~3x smaller per-frame activations
#   - BATCH_SIZE=4 vs 1-2 → 2-4x better GPU utilization
# Trade-off: -5-8% accuracy vs max-spec (still SOTA, MRL 1024d baseline 73.2 → ~67-69).
video_inputs = [
    {
        "video": p,
        "max_pixels": MAX_PIXELS,
        "fps": FPS,
    }
    for p in video_paths
]

# Qwen3-VL-Embedding uses the default instruction prompt automatically; we can pass
# a custom prompt via `prompt=` to model.encode for slight gains (+1-5%).
EMBED_INSTRUCTION = "Represent the given video for retrieval."

print(f"Encoding {len(video_inputs)} videos (batch={BATCH_SIZE}, dim={EMBEDDING_DIM})...")
t0 = time.time()
n_batches = 0

with torch.inference_mode():
    embeddings = model.encode(
        video_inputs,
        batch_size=BATCH_SIZE,
        prompt=EMBED_INSTRUCTION,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    n_batches = (len(video_inputs) + BATCH_SIZE - 1) // BATCH_SIZE

elapsed = time.time() - t0
print(f"\nInference: {elapsed:.1f}s  ({len(video_inputs) / elapsed:.2f} vid/s, {n_batches} batches)")
print(f"Output shape: {embeddings.shape}  dtype={embeddings.dtype}")

# Truncate to target dim (MRL — embedding models support Matryoshka Representation Learning)
if embeddings.shape[1] != EMBEDDING_DIM:
    print(f"  Truncating {embeddings.shape[1]}d → {EMBEDDING_DIM}d (MRL)")
    embeddings = embeddings[:, :EMBEDDING_DIM]
    # Re-normalize after truncation so cosine similarity stays correct
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / np.clip(norms, 1e-12, None)

In [ ]:
# Save embeddings to HDF5 — same schema as the other notebooks so qdrant_ingest.py
# can ingest this collection via the existing generic pipeline.
import h5py

hdf5_path = OUTPUT_DIR + "/embeddings.hdf5"
with h5py.File(hdf5_path, "w") as f:
    f.create_dataset("embeddings", data=embeddings.astype(np.float32))
    f.create_dataset("video_ids", data=[s.encode("utf-8") for s in video_ids])
    f.create_dataset("video_paths", data=[s.encode("utf-8") for s in video_paths])
    f.attrs["model_id"] = MODEL_ID
    f.attrs["feature_type"] = "vlm_embedding"
    f.attrs["embedding_dim"] = EMBEDDING_DIM
    f.attrs["num_videos"] = len(video_ids)
    f.attrs["fps"] = FPS
    f.attrs["max_pixels"] = MAX_PIXELS

print(f"[OK] Saved embeddings.hdf5 ({embeddings.nbytes / 1024**2:.2f} MB)")

# Free memory (Kaggle session can be killed if RAM > 16 GB)
del embeddings
import gc; gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
print("[OK] Pipeline complete")